In [ ]:
#| default_exp analyze

# analyze

> Story analysis step: extract characters, locations, and themes from the source text.
>
> Long stories are processed in chunks with a rolling context window so character and
> location definitions remain consistent across the full text. Results are saved to
> `analysis.json` and resumed automatically if the file already exists.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json
from pathlib import Path
from typing import Any

from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn

from manhualizer.config import PipelineConfig
from manhualizer.llm import LLMClient, chunk_story
from manhualizer.models import Character, CharacterArc, Location, StoryAnalysis
from manhualizer.prompts import TemplateSet

_console = Console()

## Parsing LLM Output

In [ ]:
#| export
def _parse_character(data: dict) -> Character:
    arcs = [
        CharacterArc(phase=a.get("phase", ""), description=a.get("description", ""))
        for a in data.get("arcs", [])
    ]
    return Character(
        name=data.get("name", "Unknown"),
        aliases=data.get("aliases", []),
        physical_description=data.get("physical_description", ""),
        personality=data.get("personality", ""),
        arcs=arcs,
        reference_image_prompt=data.get("reference_image_prompt", ""),
    )


def _parse_location(data: dict) -> Location:
    return Location(
        name=data.get("name", "Unknown"),
        description=data.get("description", ""),
        visual_prompt=data.get("visual_prompt", ""),
        atmosphere=data.get("atmosphere", ""),
    )


def _parse_analysis(data: dict, source_chunks: list[str]) -> StoryAnalysis:
    return StoryAnalysis(
        title=data.get("title", "Untitled"),
        synopsis=data.get("synopsis", ""),
        characters=[_parse_character(c) for c in data.get("characters", [])],
        locations=[_parse_location(l) for l in data.get("locations", [])],
        themes=data.get("themes", []),
        source_chunks=source_chunks,
    )

## Merging Partial Analyses

In [ ]:
#| export
def _merge_analyses_local(partials: list[dict]) -> dict:
    """Merge partial analysis dicts locally without an extra LLM call.

    Characters and locations are deduplicated by name (case-insensitive).
    Later entries take precedence for scalar fields; arcs and aliases are combined.
    """
    merged: dict[str, Any] = {
        "title": "",
        "synopsis": "",
        "characters": {},   # name → dict
        "locations": {},    # name → dict
        "themes": [],
    }

    for p in partials:
        if p.get("title") and not merged["title"]:
            merged["title"] = p["title"]
        if p.get("synopsis"):
            merged["synopsis"] = p["synopsis"]  # last wins (most complete)

        for theme in p.get("themes", []):
            if theme not in merged["themes"]:
                merged["themes"].append(theme)

        for char in p.get("characters", []):
            key = char.get("name", "").lower()
            if key not in merged["characters"]:
                merged["characters"][key] = dict(char)
            else:
                existing = merged["characters"][key]
                # Update scalar fields if the new entry has better data
                for field in ("physical_description", "personality", "reference_image_prompt"):
                    if char.get(field) and len(char[field]) > len(existing.get(field, "")):
                        existing[field] = char[field]
                # Combine aliases
                seen_aliases = set(existing.get("aliases", []))
                for alias in char.get("aliases", []):
                    if alias not in seen_aliases:
                        existing.setdefault("aliases", []).append(alias)
                        seen_aliases.add(alias)
                # Combine arcs
                seen_phases = {a["phase"] for a in existing.get("arcs", [])}
                for arc in char.get("arcs", []):
                    if arc.get("phase") not in seen_phases:
                        existing.setdefault("arcs", []).append(arc)
                        seen_phases.add(arc["phase"])

        for loc in p.get("locations", []):
            key = loc.get("name", "").lower()
            if key not in merged["locations"]:
                merged["locations"][key] = dict(loc)
            else:
                existing = merged["locations"][key]
                for field in ("description", "visual_prompt", "atmosphere"):
                    if loc.get(field) and len(loc[field]) > len(existing.get(field, "")):
                        existing[field] = loc[field]

    merged["characters"] = list(merged["characters"].values())
    merged["locations"] = list(merged["locations"].values())
    return merged


def _merge_analyses_llm(partials: list[dict], llm: LLMClient) -> dict:
    """Merge partial analyses using an LLM call for higher quality consolidation."""
    return llm.complete_from_template(
        "analyze.yml",
        "merge_prompt",
        as_json=True,
        partial_analyses=json.dumps(partials, ensure_ascii=False, indent=2),
    )

## Main Analyze Function

In [ ]:
#| export
def analyze_story(
    story_text: str,
    llm: LLMClient,
    config: PipelineConfig,
    output_path: Path,
) -> StoryAnalysis:
    """Analyze a story and extract characters, locations and themes.

    If `output_path` already exists and `config.resume` is True, the saved
    result is returned immediately without any LLM calls.

    For stories longer than `config.max_chunk_tokens` tokens, the text is
    split into chunks and analyzed iteratively. Each chunk receives the
    accumulated prior analysis as context so character definitions remain
    consistent. Partial results are merged locally (fast path) and, for
    multi-chunk stories, optionally consolidated by the LLM.

    Args:
        story_text: Full raw story text.
        llm: Configured LLMClient.
        config: PipelineConfig (uses max_chunk_tokens and resume).
        output_path: Where to save the resulting analysis.json.

    Returns:
        StoryAnalysis populated with characters, locations and themes.
    """
    output_path = Path(output_path)

    # Resume: return saved result if available
    if config.resume and output_path.exists():
        _console.print(f"[dim]analyze: resuming from {output_path}[/dim]")
        data = json.loads(output_path.read_text())
        return StoryAnalysis.model_validate(data)

    chunks = chunk_story(story_text, max_tokens=config.max_chunk_tokens)
    _console.print(f"[bold]analyze:[/bold] processing {len(chunks)} chunk(s)")

    partials: list[dict] = []
    prior_analysis = "{}"

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        console=_console,
        transient=True,
    ) as progress:
        task = progress.add_task("Analyzing...", total=len(chunks))

        for i, chunk in enumerate(chunks):
            progress.update(task, description=f"Analyzing chunk {i + 1}/{len(chunks)}...")
            result = llm.complete_from_template(
                "analyze.yml",
                "analyze_prompt",
                as_json=True,
                story_chunk=chunk,
                prior_analysis=prior_analysis,
            )
            partials.append(result)
            # Pass latest partial as rolling context for next chunk
            prior_analysis = json.dumps(result, ensure_ascii=False)
            progress.advance(task)

    # Merge
    if len(partials) == 1:
        merged = partials[0]
    else:
        _console.print("[bold]analyze:[/bold] merging partial analyses")
        merged = _merge_analyses_local(partials)
        # For multi-chunk stories, do an LLM consolidation pass for quality
        try:
            merged = _merge_analyses_llm(partials, llm)
        except Exception as e:
            _console.print(f"[yellow]analyze: LLM merge failed, using local merge ({e})[/yellow]")

    analysis = _parse_analysis(merged, chunks)

    # Save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        analysis.model_dump_json(indent=2, exclude_none=True)
    )
    _console.print(f"[green]analyze: saved to {output_path}[/green]")
    _console.print(
        f"  {len(analysis.characters)} character(s), "
        f"{len(analysis.locations)} location(s), "
        f"{len(analysis.themes)} theme(s)"
    )

    return analysis

## Tests (no API calls)

In [ ]:
from manhualizer.analyze import _merge_analyses_local, _parse_analysis

partial_a = {
    "title": "The Dragon's Gift",
    "synopsis": "A young farmer discovers a dragon's egg.",
    "characters": [{
        "name": "Wei Chen",
        "aliases": [],
        "physical_description": "Tall young man with short black hair",
        "personality": "Determined",
        "arcs": [{"phase": "beginning", "description": "Ordinary farmer"}],
        "reference_image_prompt": "young Chinese man, short black hair",
    }],
    "locations": [{
        "name": "Village",
        "description": "Small farming village",
        "visual_prompt": "chinese village, rice paddies",
        "atmosphere": "warm, golden light",
    }],
    "themes": ["destiny", "courage"],
}

partial_b = {
    "title": "The Dragon's Gift",
    "synopsis": "Wei Chen hatches the egg and befriends the dragon.",
    "characters": [{
        "name": "Wei Chen",
        "aliases": ["The Dragon Keeper"],
        "physical_description": "Tall young man with short black hair, now wearing dragon-scale armour",
        "personality": "Determined and compassionate",
        "arcs": [{"phase": "middle", "description": "Bonded with dragon"}],
        "reference_image_prompt": "young Chinese man, short black hair, dragon-scale armour",
    }],
    "locations": [{
        "name": "Mountain Cave",
        "description": "A vast cave with crystal formations",
        "visual_prompt": "giant cave, glowing crystals, dragon",
        "atmosphere": "mysterious, blue glow",
    }],
    "themes": ["destiny", "friendship"],
}

merged = _merge_analyses_local([partial_a, partial_b])

# Character deduplication
assert len(merged["characters"]) == 1, f"Expected 1 character, got {len(merged['characters'])}"
char = merged["characters"][0]
assert char["name"] == "Wei Chen"
assert len(char["arcs"]) == 2  # beginning + middle combined
assert "The Dragon Keeper" in char["aliases"]
# Longer description wins
assert "armour" in char["physical_description"]

# Location deduplication
assert len(merged["locations"]) == 2  # village + mountain cave

# Theme deduplication
assert sorted(merged["themes"]) == ["courage", "destiny", "friendship"]

# Parse to StoryAnalysis
analysis = _parse_analysis(merged, ["chunk1", "chunk2"])
assert analysis.title == "The Dragon's Gift"
assert len(analysis.characters) == 1
assert len(analysis.locations) == 2
assert len(analysis.source_chunks) == 2

# Serialise round-trip
as_json = analysis.model_dump_json()
from manhualizer.models import StoryAnalysis
restored = StoryAnalysis.model_validate_json(as_json)
assert restored.characters[0].name == "Wei Chen"

print("analyze: all tests passed")

In [ ]:
# Resume test: analyze_story returns saved result without LLM calls
import tempfile, json
from pathlib import Path
from manhualizer.analyze import analyze_story
from manhualizer.config import PipelineConfig

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "analysis.json"
    out.write_text(analysis.model_dump_json())

    cfg = PipelineConfig(resume=True)
    # Pass None as llm — if resume works, it won't be called
    result = analyze_story("any text", llm=None, config=cfg, output_path=out)
    assert result.title == "The Dragon's Gift"

print("analyze: resume test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()